# Stage 11 — DeepScoresV2 Dense Stage 9A Symbol-Region Preservation Evaluation v2

This is a **non-tuning, non-training** evaluation of the frozen `best.pt` checkpoint on the official DeepScoresV2 Dense held-out split.

DeepScoresV2 Dense uses its native annotation schema: `images` is a list, `annotations` is an ID-keyed dictionary, each image references annotation IDs through `ann_ids`, categories are ID-keyed, and `a_bbox` is interpreted as `[left, top, right, bottom]`.

The assay evaluates annotation-centered symbol image regions after **geometry-preserving** camera-like degradation. It does not create an optimizer, perform backpropagation, retune the model, or claim OMR correctness / musical truth.

CUDA is preferred; CPU is supported. Progress is atomically persisted to Google Drive and resumed after runtime interruption. No keep-alive or Colab idle-limit bypass is used.


In [ ]:
from pathlib import Path
import hashlib, json, tarfile, random, math, io, os, time
from google.colab import drive
drive.mount("/content/drive")

A=Path("/content/drive/MyDrive/ST_SCORE_RESTORE_STAGE11_TRAINING_DATA/ds2_dense.tar.gz")
O=Path("/content/drive/MyDrive/ST_SCORE_RESTORE_STAGE11_TRAINING_OUTPUT/deepscoresv2_dense_residual_unet_v1")
BEST=O/"best.pt"
X=Path("/content/st_score_restore_deepscoresv2_dense")
PROGRESS=O/"stage9a_symbol_region_progress.v2.json"
FINAL=O/"stage9a_symbol_region_final_evidence.v2.json"
MD5="7237318e381e6e0848ec30eb82decb83"
SIZE=741814529
EXPECTED_BEST_SHA256="08b279161a9e8c4bd37376da221ecb4e07130724254ccf7d9591c8d32f368683"
EXPECTED_CONFIG="deff0f1270009839e234608dd9967038e1228a6b2b034e26059ac2f8cbfd0f80"
SEED=20260907
PER_CATEGORY=8
MAX_SAMPLES=1024
PATCH=128
if not A.exists():
    matches=list(Path("/content/drive/MyDrive").rglob("ds2_dense.tar.gz"))
    if len(matches)!=1: raise FileNotFoundError(f"Expected one ds2_dense.tar.gz, found {len(matches)}")
    A=matches[0]
if not BEST.exists(): raise FileNotFoundError(BEST)
if A.stat().st_size!=SIZE: raise RuntimeError("Archive size mismatch")
def file_hash(path, algorithm):
    h=hashlib.new(algorithm)
    with path.open("rb") as f:
        while b:=f.read(8<<20): h.update(b)
    return h.hexdigest()
if file_hash(A,"md5")!=MD5: raise RuntimeError("Archive MD5 mismatch")
if file_hash(BEST,"sha256")!=EXPECTED_BEST_SHA256: raise RuntimeError("best.pt SHA256 mismatch")
def safe_extract():
    X.mkdir(parents=True,exist_ok=True); base=X.resolve()
    with tarfile.open(A,"r:gz") as t:
        members=t.getmembers()
        for m in members:
            p=m.name.replace("\\","/")
            if not p or p.startswith("/") or ".." in Path(p).parts or m.issym() or m.islnk() or m.isdev(): raise RuntimeError(f"unsafe tar member: {m.name}")
            q=(X/p).resolve()
            if q!=base and base not in q.parents: raise RuntimeError(f"tar escape: {m.name}")
        t.extractall(X)
if not X.exists() or not any(X.iterdir()): safe_extract()
tests=list(X.rglob("deepscores_test.json"))
if len(tests)!=1: raise RuntimeError(f"Expected one deepscores_test.json, found {len(tests)}")
TEST_JSON=tests[0]; R=TEST_JSON.parent; payload=json.loads(TEST_JSON.read_text(encoding="utf-8"))
images=payload.get("images"); annotations=payload.get("annotations"); categories=payload.get("categories")
if not isinstance(images,list): raise RuntimeError(f"Unsupported DeepScores images structure: {type(images).__name__}")
if not isinstance(annotations,dict): raise RuntimeError(f"Unsupported DeepScores annotations structure: expected dict, got {type(annotations).__name__}")
if not isinstance(categories,dict): raise RuntimeError(f"Unsupported DeepScores categories structure: expected dict, got {type(categories).__name__}")
print("DeepScores native schema:","images=list",len(images),"annotations=dict",len(annotations),"categories=dict",len(categories))
def image_name(row): return row.get("filename") or row.get("file_name") or row.get("img_name") or row.get("name")
def image_key(row, fallback_index):
    for k in ("id","img_id","image_id"):
        if row.get(k) is not None: return str(row[k])
    name=image_name(row); return str(name) if name else f"row:{fallback_index}"
def category_ids(annotation):
    raw=annotation.get("cat_id")
    if raw is None: raw=annotation.get("category_id")
    if raw is None: raw=annotation.get("categoryId")
    if isinstance(raw,(list,tuple)): return [str(x) for x in raw if x is not None]
    return [] if raw is None else [str(raw)]
def category_name(cid):
    row=categories.get(str(cid), categories.get(cid))
    if isinstance(row,dict): return str(row.get("name") or row.get("label") or cid)
    return str(row) if row is not None else str(cid)
def native_bbox(annotation):
    b=annotation.get("a_bbox")
    if isinstance(b,(list,tuple)) and len(b)>=4:
        left,top,right,bottom=[float(x) for x in b[:4]]
        if right>left and bottom>top: return (left,top,right-left,bottom-top)
    b=annotation.get("bbox")
    if isinstance(b,(list,tuple)) and len(b)>=4:
        x,y,w,h=[float(x) for x in b[:4]]
        if w>0 and h>0: return (x,y,w,h)
    return None
def image_path(name):
    for p in (R/"images"/name,R/name,TEST_JSON.parent/"images"/name):
        if p.exists(): return p
    matches=list(R.rglob(Path(name).name))
    if len(matches)==1: return matches[0]
    raise FileNotFoundError(name)
from PIL import Image
image_rows={}; records=[]
for row_index,row in enumerate(images):
    if not isinstance(row,dict): continue
    name=image_name(row)
    if not name: continue
    iid=image_key(row,row_index); image_rows[iid]=row; ann_ids=row.get("ann_ids") or []
    if not isinstance(ann_ids,(list,tuple)): raise RuntimeError(f"DeepScores image ann_ids is not a list for {name}")
    p=image_path(str(name))
    with Image.open(p) as im: iw,ih=im.width,im.height
    for ann_id in ann_ids:
        a=annotations.get(str(ann_id), annotations.get(ann_id))
        if not isinstance(a,dict): continue
        bbox=native_bbox(a)
        if bbox is None: continue
        x,y,w,h=bbox
        if x<0 or y<0 or w<2 or h<2 or x+w>iw+2 or y+h>ih+2: continue
        cids=category_ids(a)
        if not cids: continue
        cid=cids[0]; token=f"{cid}:{iid}:{ann_id}:{bbox}"; key=hashlib.sha256(f"{SEED}:{token}".encode()).hexdigest()
        records.append({"key":key,"annotationId":str(ann_id),"imageId":iid,"imageName":str(name),"categoryId":cid,"categoryName":category_name(cid),"bbox":bbox})
if len(image_rows)!=352: raise RuntimeError(f"Expected 352 official held-out images, got {len(image_rows)}")
if not records: raise RuntimeError("No valid DeepScores native annotations found")
by_cat={}
for r in records: by_cat.setdefault(r["categoryId"],[]).append(r)
selected=[]
for cid in sorted(by_cat): selected.extend(sorted(by_cat[cid],key=lambda r:r["key"])[:PER_CATEGORY])
selected=sorted(selected,key=lambda r:hashlib.sha256(f"{SEED}:{r['categoryId']}:{r['imageId']}:{r['annotationId']}".encode()).hexdigest())[:MAX_SAMPLES]
if not selected: raise RuntimeError("No valid symbol annotations selected")
selection_digest=hashlib.sha256(json.dumps(selected,sort_keys=True,separators=(",",":")).encode()).hexdigest()
print("Stage9A samples:",len(selected),"categories:",len({r["categoryId"] for r in selected}),"selection:",selection_digest[:12])


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np
from PIL import Image,ImageFilter,ImageOps
from torchvision.transforms import functional as TF
D=torch.device("cuda" if torch.cuda.is_available() else "cpu"); DEVICE_NAME=torch.cuda.get_device_name(0) if D.type=="cuda" else "CPU"
print("Evaluation device:",DEVICE_NAME)
class C(nn.Module):
    def __init__(self,a,b): super().__init__(); self.n=nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.GroupNorm(8,b),nn.SiLU(),nn.Conv2d(b,b,3,padding=1),nn.GroupNorm(8,b),nn.SiLU())
    def forward(self,x): return self.n(x)
class UNet(nn.Module):
    def __init__(self,b=32): super().__init__(); self.e1=C(1,b); self.e2=C(b,2*b); self.e3=C(2*b,4*b); self.mid=C(4*b,8*b); self.d3=C(12*b,4*b); self.d2=C(6*b,2*b); self.d1=C(3*b,b); self.o=nn.Conv2d(b,1,1)
    def forward(self,x):
        a=self.e1(x); b=self.e2(F.max_pool2d(a,2)); c=self.e3(F.max_pool2d(b,2)); d=self.mid(F.max_pool2d(c,2)); d=self.d3(torch.cat([F.interpolate(d,size=c.shape[-2:],mode="bilinear",align_corners=False),c],1)); d=self.d2(torch.cat([F.interpolate(d,size=b.shape[-2:],mode="bilinear",align_corners=False),b],1)); d=self.d1(torch.cat([F.interpolate(d,size=a.shape[-2:],mode="bilinear",align_corners=False),a],1)); return torch.clamp(x+torch.tanh(self.o(d))*.5,0,1)
m=UNet().to(D); ck=torch.load(BEST,map_location=D)
if ck.get("md5")!=MD5 or ck.get("cfg")!=EXPECTED_CONFIG or ck.get("epoch")!=19: raise RuntimeError("Checkpoint identity mismatch")
m.load_state_dict(ck["model"]); m.eval()
sx=torch.tensor([[-1.,0,1],[-2,0,2],[-1,0,1]],device=D).view(1,1,3,3); sy=sx.transpose(2,3)
def edge(x): return torch.sqrt(F.conv2d(x,sx,padding=1)**2+F.conv2d(x,sy,padding=1)**2+1e-6)
def state_sha256(model):
    h=hashlib.sha256()
    for name,t in sorted(model.state_dict().items()): h.update(name.encode()); h.update(t.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()
WEIGHTS_BEFORE=state_sha256(m)
def centered_symbol_patch(im,bbox):
    x,y,w,h=bbox; cx=x+w/2; cy=y+h/2; side=max(48.0,4.0*max(w,h)); left=int(math.floor(cx-side/2)); top=int(math.floor(cy-side/2)); right=int(math.ceil(cx+side/2)); bottom=int(math.ceil(cy+side/2)); pad_l=max(0,-left); pad_t=max(0,-top); pad_r=max(0,right-im.width); pad_b=max(0,bottom-im.height)
    if any((pad_l,pad_t,pad_r,pad_b)): im=ImageOps.expand(im,border=(pad_l,pad_t,pad_r,pad_b),fill=255); left+=pad_l; right+=pad_l; top+=pad_t; bottom+=pad_t
    return im.crop((left,top,right,bottom)).resize((PATCH,PATCH),Image.Resampling.BICUBIC)
def rng_for(row): return random.Random(int.from_bytes(hashlib.sha256(f"{SEED}:stage9a:{row['categoryId']}:{row['imageId']}:{row['annotationId']}".encode()).digest()[:8],"big"))
def degrade_symbol_patch(im,r):
    im=TF.adjust_brightness(im,r.uniform(.78,1.2)); im=TF.adjust_contrast(im,r.uniform(.82,1.18)); im=im.filter(ImageFilter.GaussianBlur(r.uniform(.15,1.75))); a=np.asarray(im).astype(np.float32)/255.; yy,xx=np.mgrid[:a.shape[0],:a.shape[1]]; cx,cy=r.uniform(0,a.shape[1]),r.uniform(0,a.shape[0]); z=np.sqrt((xx-cx)**2+(yy-cy)**2); z/=max(z.max(),1); a*=1-r.uniform(0,.22)*(1-z); a+=np.random.default_rng(r.randrange(2**32)).normal(0,r.uniform(.002,.03),a.shape); a=np.clip(a,0,1); out=Image.fromarray((a*255).astype("uint8")); buf=io.BytesIO(); out.save(buf,"JPEG",quality=r.randint(52,95)); buf.seek(0); return Image.open(buf).convert("L")
def ten(im): return torch.from_numpy(np.asarray(im,dtype=np.float32)/255.).unsqueeze(0).unsqueeze(0)
def metrics(pred,target):
    pixel=F.l1_loss(pred,target).item(); edge_loss=F.l1_loss(edge(pred),edge(target)).item(); mse=F.mse_loss(pred,target).item(); ink=(target<0.75); white=(target>0.95); pred_ink=(pred<0.85); ink_recall=((pred_ink & ink).float().sum()/ink.float().sum().clamp_min(1)).item(); false_ink=(((pred<0.80)&white).float().sum()/white.float().sum().clamp_min(1)).item(); return {"pixelL1":pixel,"edgeL1":edge_loss,"mse":mse,"inkRecall":ink_recall,"falseInkRate":false_ink}
def zero_agg(): return {"count":0,"baseline":{"pixelL1":0.,"edgeL1":0.,"mse":0.,"inkRecall":0.,"falseInkRate":0.},"restored":{"pixelL1":0.,"edgeL1":0.,"mse":0.,"inkRecall":0.,"falseInkRate":0.}}
def add_metrics(agg,side,vals):
    for k,v in vals.items(): agg[side][k]+=float(v)
def atomic_json(path,payload): path.parent.mkdir(parents=True,exist_ok=True); tmp=path.with_name(path.name+".tmp"); tmp.write_text(json.dumps(payload,indent=2,sort_keys=True)); os.replace(tmp,path)


In [ ]:
IDENTITY={"schemaVersion":"stage11.stage9a-symbol-region-progress.v2","datasetId":"deepscoresv2.dense.v2","datasetMd5":MD5,"checkpointSha256":EXPECTED_BEST_SHA256,"configSha256":EXPECTED_CONFIG,"selectionSha256":selection_digest,"selectedSamples":len(selected),"annotationSchema":"deepscores-native-v2:images-list+ann_ids+annotations-dict+a_bbox-xyxy","geometryPreservingAssay":True}
start=0; agg=zero_agg()
if PROGRESS.exists():
    prior=json.loads(PROGRESS.read_text())
    for k,v in IDENTITY.items():
        if prior.get(k)!=v: raise RuntimeError(f"Progress identity mismatch for {k}")
    start=int(prior.get("nextIndex",0)); agg=prior.get("aggregate") or zero_agg(); print("Resuming Stage9A at",start,"of",len(selected))
with torch.inference_mode():
    for i in range(start,len(selected)):
        row=selected[i]; p=image_path(row["imageName"])
        with Image.open(p) as raw: clean=centered_symbol_patch(raw.convert("L"),row["bbox"])
        degraded=degrade_symbol_patch(clean,rng_for(row)); target=ten(clean).to(D); source=ten(degraded).to(D); restored=m(source); b=metrics(source,target); r=metrics(restored,target); add_metrics(agg,"baseline",b); add_metrics(agg,"restored",r); agg["count"]+=1
        if (i+1)%16==0 or i+1==len(selected): atomic_json(PROGRESS,{**IDENTITY,"nextIndex":i+1,"aggregate":agg,"updatedAtUnix":time.time()}); print("saved",i+1,"/",len(selected))
if agg["count"]!=len(selected): raise RuntimeError(f"Evaluation incomplete: {agg['count']} != {len(selected)}")
def avg(side): return {k:v/agg["count"] for k,v in agg[side].items()}
baseline=avg("baseline"); restored=avg("restored"); WEIGHTS_AFTER=state_sha256(m); weights_mutated=(WEIGHTS_AFTER!=WEIGHTS_BEFORE)
improvement={"pixelL1Percent":100*(baseline["pixelL1"]-restored["pixelL1"])/max(baseline["pixelL1"],1e-12),"edgeL1Percent":100*(baseline["edgeL1"]-restored["edgeL1"])/max(baseline["edgeL1"],1e-12),"msePercent":100*(baseline["mse"]-restored["mse"])/max(baseline["mse"],1e-12),"inkRecallDelta":restored["inkRecall"]-baseline["inkRecall"],"falseInkRateDelta":restored["falseInkRate"]-baseline["falseInkRate"]}
stage9a_pass=(not weights_mutated and restored["pixelL1"]<baseline["pixelL1"] and restored["edgeL1"]<baseline["edgeL1"] and restored["mse"]<baseline["mse"] and improvement["inkRecallDelta"]>=-0.02 and improvement["falseInkRateDelta"]<=0.02)
evidence={"schemaVersion":"stage11.stage9a-symbol-region-final.v2","datasetId":"deepscoresv2.dense.v2","datasetMd5":MD5,"checkpointSha256":EXPECTED_BEST_SHA256,"checkpointWeightsBeforeSha256":WEIGHTS_BEFORE,"checkpointWeightsAfterSha256":WEIGHTS_AFTER,"weightsMutated":weights_mutated,"executionDevice":DEVICE_NAME,"officialHeldOutImages":352,"selectedSymbolRegions":len(selected),"categoryCount":len({r["categoryId"] for r in selected}),"selectionSha256":selection_digest,"annotationSchema":IDENTITY["annotationSchema"],"geometryPreservingAssay":True,"optimizerCreated":False,"backpropagationExecuted":False,"heldOutUsedForTraining":False,"heldOutUsedForTuning":False,"baseline":baseline,"restored":restored,"improvement":improvement,"stage9aPreservationPass":stage9a_pass,"omrCorrectnessImplied":False,"musicalTruthImplied":False,"automaticFinalSelectionAuthorized":False,"finalStage11Pass":False}
atomic_json(FINAL,evidence); print(json.dumps(evidence,indent=2,sort_keys=True)); print("Stage 9A preservation proxy:","PASS" if stage9a_pass else "REVIEW_REQUIRED")
